# 2. Generate Australian Centre for Advanced Photovoltaics (ACAP), Energy Data Platform (ACAP-EDP) Different-Aggregation Datasets

## Purpose
Generate 12 weather-enhanced PyNNLF datasets from the valid per-household checkpoints produced by notebook `1_build_acap_edp_site_30min_checkpoints.ipynb`.

Important modelling choice: this notebook intentionally forward-fills each household's 30-minute checkpoint before aggregating households. This does not exactly reproduce the historical `ds10`/`ds11` group-level aggregation order, but it gives each sampled household/group a complete profile before summation, which is the desired behaviour for this aggregation-level experiment.

## Inputs
- `SITE_LIST_PATH`
- `VALID_SITES_PATH`
- `WEATHER_PATH`

## Run Flow
1. Setup And Paths
2. Load Valid Checkpoint Sites And Weather
3. Create Reproducible Sample Design And Display Chosen Households
4. Forward-Fill Households, Then Aggregate And Export
5. Debug Notes

## Outputs
- `AUDIT_DIR / "acap_edp_aggregation_sample_design.csv"`
- `AUDIT_DIR / "acap_edp_aggregation_sample_membership.csv"`
- `AUDIT_DIR / "acap_edp_aggregation_sample_membership_with_metadata.csv"`
- `DATA_EXPLORATION_DIR / "acap_edp_aggregation_sample_design.csv"`
- `DATA_EXPLORATION_DIR / "acap_edp_aggregation_sample_membership.csv"`
- `DATA_EXPLORATION_DIR / "acap_edp_aggregation_sample_membership_with_metadata.csv"`
- `AUDIT_DIR / "acap_edp_aggregation_dataset_export_summary.csv"`
- `DATA_EXPLORATION_DIR / "acap_edp_aggregation_dataset_export_summary.csv"`

## 1. Setup And Paths

In [ ]:
import os
from pathlib import Path
import sys


def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")


PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")

import pandas as pd
import numpy as np

def raw_data_root():
    """Locate the local source data tree, which is not distributed with this repository.

    Set the PYNNLF_RAW_DATA_DIR environment variable to the directory holding the
    "1. raw", "2. processed" and "3. cleaned" folders before running this notebook.

    Returns:
        Path: root of the local source data tree.
    """
    root = os.environ.get("PYNNLF_RAW_DATA_DIR")
    if not root:
        raise RuntimeError(
            "PYNNLF_RAW_DATA_DIR is not set. Point it at your local source data "
            "directory; see the Data section of the repository README."
        )
    return Path(root)


RAW_DATA_ROOT = raw_data_root()

PROCESSED_DIR = RAW_DATA_ROOT / "2. processed"
CLEANED_WORKSPACE = RAW_DATA_ROOT / "3. cleaned" / "ACAP_EDP_different_aggregation"
SITE_LIST_PATH = PROCESSED_DIR / "acap_edp_cluster_2_2_3years.xlsx"
WEATHER_PATH = PROCESSED_DIR / "acap_edp_weather_data.csv"
SITE_30MIN_DIR = CLEANED_WORKSPACE / "checkpoints" / "site_30min"
AUDIT_DIR = CLEANED_WORKSPACE / "audits"
VALID_SITES_PATH = AUDIT_DIR / "acap_edp_valid_checkpoint_sites.csv"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_DATA_DIR = REPO_ROOT / "data"
PUBLICATION_DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results" / "03_acap_edp_aggregation_level"
DATA_EXPLORATION_DIR = RESULTS_DIR / "01_data_exploration"
for directory in [RESULTS_DIR, DATA_EXPLORATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

START = pd.Timestamp("2021-07-01 00:00:00")
END_30MIN = pd.Timestamp("2024-06-30 23:30:00")
INDEX_30MIN = pd.date_range(START, END_30MIN, freq="30min", name="datetime")
RANDOM_SEED = 20260521
SAMPLES_PER_LEVEL = 3
AGGREGATION_LEVELS = [1, 10, 100, 1000]
MIN_VALID_SITES = 100

for path in [SITE_LIST_PATH, WEATHER_PATH, SITE_30MIN_DIR, VALID_SITES_PATH, ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)
print(f"Cleaned ACAP-EDP workspace: {CLEANED_WORKSPACE}")
print(f"Valid checkpoint site list: {VALID_SITES_PATH}")


## 2. Load Valid Checkpoint Sites And Weather

This notebook samples only from the valid checkpoint sites produced by notebook `2.1`. Sites excluded during checkpointing are not sampled. The weather variables are the same three variables used by the existing ACAP-EDP weather dataset.


In [ ]:
site_list = pd.read_excel(SITE_LIST_PATH)
site_list["edp_site_id"] = site_list["edp_site_id"].astype(str).str.strip()
valid_sites = pd.read_csv(VALID_SITES_PATH)
valid_sites["edp_site_id"] = valid_sites["edp_site_id"].astype(str).str.strip()
valid_sites = valid_sites.loc[valid_sites["status"].eq("valid")].copy()
site_ids = valid_sites["edp_site_id"].tolist()

if len(site_ids) != len(set(site_ids)):
    raise ValueError("Valid checkpoint site list contains duplicate site IDs")
if len(site_ids) < MIN_VALID_SITES:
    raise ValueError(f"Only {len(site_ids)} valid checkpoint sites are available; at least {MIN_VALID_SITES} are needed for 100hh sampling")

missing_checkpoint_files = [site_id for site_id in site_ids if not (SITE_30MIN_DIR / f"{site_id}_30min.parquet").exists()]
if missing_checkpoint_files:
    raise FileNotFoundError("Valid site list references missing checkpoint files. First missing IDs: " + str(missing_checkpoint_files[:10]))

site_metadata = valid_sites.merge(
    site_list[["edp_site_id", "postcode", "state", "lat_deg", "long_deg"]],
    on="edp_site_id",
    how="left",
    suffixes=("", "_site_list"),
)
weather = pd.read_csv(WEATHER_PATH, parse_dates=["datetime"]).set_index("datetime").reindex(INDEX_30MIN)
expected_weather_columns = ["air_temperature_in_degrees_c", "relative_humidity_in_percentage", "wind_speed_in_km_h"]
if list(weather.columns) != expected_weather_columns:
    raise ValueError(f"Unexpected weather columns: {list(weather.columns)}")
if weather.isna().any().any():
    raise ValueError("Weather data contains missing values after reindexing")

print(f"Valid checkpoint sites available for sampling: {len(site_ids)}")
display(site_metadata.head())
display(weather.head())


## 3. Create Reproducible Sample Design And Display Chosen Households

This section creates 3 random samples per aggregation level:

- `1hh`: 3 single-household samples.
- `10hh`: 3 groups of 10 households, sampled without replacement within each group.
- `100hh`: 3 groups of 100 households, sampled without replacement within each group.
- `1000hh`: 3 synthetic bootstrap groups, sampled with replacement from the valid checkpoint sites.

Groups may overlap across samples. No duplicates are allowed within the 1/10/100hh groups. The 1000hh level intentionally allows repeated household IDs and stores those repetitions as integer weights.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
dataset_rows = []
membership_rows = []
next_dataset_no = 25
expected_dataset_count = SAMPLES_PER_LEVEL * len(AGGREGATION_LEVELS)
expected_dataset_ids = [f"ds{i}" for i in range(25, 25 + expected_dataset_count)]

for level in AGGREGATION_LEVELS:
    for sample_no in range(1, SAMPLES_PER_LEVEL + 1):
        dataset_id = f"ds{next_dataset_no}"
        if level < 1000:
            selected = rng.choice(site_ids, size=level, replace=False).tolist()
        else:
            selected = rng.choice(site_ids, size=level, replace=True).tolist()
        counts = pd.Series(selected, name="edp_site_id").value_counts().sort_index()
        for site_id, weight in counts.items():
            membership_rows.append({"dataset_id": dataset_id, "aggregation_level_hh": level, "sample_no": sample_no, "edp_site_id": site_id, "weight": int(weight)})
        dataset_rows.append({
            "dataset_id": dataset_id,
            "dataset_no": next_dataset_no,
            "aggregation_level_hh": level,
            "sample_no": sample_no,
            "sample_label": f"{level}hh_sample{sample_no:02d}",
            "filename": f"{dataset_id}_acap-edp_{level}hh_sample{sample_no:02d}_30min_with_weather.csv",
            "unique_households": int(counts.shape[0]),
            "total_household_weight": int(counts.sum()),
        })
        next_dataset_no += 1

sample_design = pd.DataFrame(dataset_rows)
membership = pd.DataFrame(membership_rows)
if sample_design.shape[0] != expected_dataset_count or sample_design["dataset_id"].tolist() != expected_dataset_ids:
    raise ValueError(f"Sample design should contain {expected_dataset_count} datasets: {expected_dataset_ids[0]} through {expected_dataset_ids[-1]}")

membership_display = membership.merge(
    site_metadata[["edp_site_id", "postcode", "state", "lat_deg", "long_deg"]],
    on="edp_site_id",
    how="left",
).sort_values(["dataset_id", "edp_site_id"])

sample_design.to_csv(AUDIT_DIR / "acap_edp_aggregation_sample_design.csv", index=False)
membership.to_csv(AUDIT_DIR / "acap_edp_aggregation_sample_membership.csv", index=False)
membership_display.to_csv(AUDIT_DIR / "acap_edp_aggregation_sample_membership_with_metadata.csv", index=False)
sample_design.to_csv(DATA_EXPLORATION_DIR / "acap_edp_aggregation_sample_design.csv", index=False)
membership.to_csv(DATA_EXPLORATION_DIR / "acap_edp_aggregation_sample_membership.csv", index=False)
membership_display.to_csv(DATA_EXPLORATION_DIR / "acap_edp_aggregation_sample_membership_with_metadata.csv", index=False)

display(sample_design)
display(membership_display)


## 4. Forward-Fill Households, Then Aggregate And Export

For each dataset sample, the notebook loads each selected household checkpoint, reindexes it to the full 30-minute study grid, forward-fills that household series, multiplies by the household's bootstrap weight, and only then adds it to the group total.

This is intentional. It differs from the historical `ds10`/`ds11` validation in notebook 1, where the aggregate was summed at raw timestamp level before forward-fill. For this aggregation-level experiment, we want missingness handled at the household level before constructing 1hh, 10hh, 100hh, and bootstrap 1000hh groups.


In [ ]:
def load_weighted_group_series(weights: pd.Series) -> pd.Series:
    total = pd.Series(0.0, index=INDEX_30MIN, name="netload_kW")
    for site_id, weight in weights.items():
        path = SITE_30MIN_DIR / f"{site_id}_30min.parquet"
        frame = pd.read_parquet(path)
        series = frame.set_index("datetime")["netload_kW"].reindex(INDEX_30MIN)

        # Intentional order for this experiment: fill each household before aggregation.
        series = series.ffill()
        if series.isna().any():
            raise ValueError(f"{site_id}: checkpoint still has missing values after per-household forward-fill")

        total = total.add(series * int(weight), fill_value=0.0)
    return total


export_rows = []
for row in sample_design.itertuples(index=False):
    member = membership.loc[membership["dataset_id"].eq(row.dataset_id)]
    weights = member.set_index("edp_site_id")["weight"]
    netload = load_weighted_group_series(weights)
    dataset = netload.to_frame().merge(weather, left_index=True, right_index=True, how="left").reset_index()
    if dataset.shape[0] != 52608 or dataset.isna().any().any() or dataset["datetime"].duplicated().any():
        raise ValueError(f"{row.dataset_id}: exported dataset failed validation")
    for target_dir in [ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
        out_path = target_dir / row.filename
        dataset.to_csv(out_path, index=False)
        print(f"Wrote: {out_path}")
    export_rows.append({
        "dataset_id": row.dataset_id,
        "filename": row.filename,
        "aggregation_level_hh": row.aggregation_level_hh,
        "sample_no": row.sample_no,
        "rows": dataset.shape[0],
        "start": dataset["datetime"].min(),
        "end": dataset["datetime"].max(),
        "unique_households": row.unique_households,
        "total_household_weight": row.total_household_weight,
        "fill_order": "per_household_forward_fill_before_aggregation",
    })

export_summary = pd.DataFrame(export_rows)
export_summary.to_csv(AUDIT_DIR / "acap_edp_aggregation_dataset_export_summary.csv", index=False)
export_summary.to_csv(DATA_EXPLORATION_DIR / "acap_edp_aggregation_dataset_export_summary.csv", index=False)
display(export_summary)


## 5. Debug Notes

If checkpoint files are missing, run notebook `1_build_acap_edp_site_30min_checkpoints.ipynb` first. Re-running this notebook overwrites CSV outputs deterministically because the random seed and sample count are fixed.

Expected outputs are 12 datasets: `ds25` through `ds36`. The chosen household membership is displayed above and saved as `acap_edp_aggregation_sample_membership_with_metadata.csv`.
